# Importation des librairies

In [1]:
import os

In [2]:
os.environ["KERAS_BACKEND"]="torch"
import torch
import gc
import json
import collections
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import time
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import tensorflow as tf
# TensorFlow ne sert qu'au chargement des données (tf.data, Tokenizer) : on lui cache le GPU
# pour qu'il ne réserve pas toute la mémoire vidéo, laissée à Torch pour l'entraînement
tf.config.set_visible_devices([], "GPU")
import keras
from keras import layers
from keras.applications.inception_v3 import InceptionV3
from sklearn.utils import shuffle

# Keras (backend Torch) utilise automatiquement le GPU s'il est disponible
print("Entraînement sur :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (aucun GPU détecté)")

# Chemins et imports
Le notebook sélectionne le backend Torch avant d’importer Keras. Il cherche le dossier models à partir du répertoire de travail, puis définit les chemins des modèles et des données.

In [3]:
PROJECT_DIR = os.path.abspath(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CLASSIFIER_WEIGHTS = os.path.join(MODELS_DIR, "model_full.weights.h5")
DENOISER_MODEL = os.path.join(MODELS_DIR, "unet_denoiser_livrable2.keras")
DATASETS_DIR = os.path.join(PROJECT_DIR, "datasets", "Livrable 3")
COCO_DIR = os.path.join(DATASETS_DIR, "train2014")
PHOTOS_DIR = os.path.join(DATASETS_DIR, "photos")
DENOISED_DIR = os.path.join(DATASETS_DIR, "denoised_images")
# Caractéristiques InceptionV3 déjà calculées (64, 2048) : remplace DENOISED_DIR pour gagner du temps
FEATURES_DIR = os.path.join(DATASETS_DIR, "features_extracted")
ANNOTATIONS_DIR = os.path.join(DATASETS_DIR, "annotations")
ANNOTATION_FILE = os.path.join(ANNOTATIONS_DIR, "captions_train2014.json")

# Etape 1 : classification Photo / Non-photo
Le modèle du livrable 1 reconnaît cinq classes : Painting, Photo, Schematics, Sketch et Text. Il faut donc garder ses cinq sorties : l’étape est dite binaire parce qu’on transforme ensuite sa prédiction en deux groupes, Photo et Non-photo.

Pour chaque image, le code :

la redimensionne en 128 × 128 ;
la passe au modèle, qui divise ses pixels par 255 grâce à sa couche Rescaling ;
récupère la probabilité associée à Photo ;
la compare au seuil 0,2301, défini dans le livrable 1.
Si la probabilité atteint le seuil, l’image est considérée comme une photo. Sinon, elle reste dans son dossier d’origine. Le code déplace les photos retenues dans datasets/photos — il ne les copie pas — mais ne déplace pas et ne supprime pas les autres images.

In [4]:
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
NUM_CLASSES = len(CLASS_NAMES)
PHOTO_CLASS_INDEX = CLASS_NAMES.index("Photo")
THRESHOLD = 0.2301

In [5]:
def load_classifier_model():
    model = keras.models.load_model(CLASSIFIER_WEIGHTS, compile=False)
    return model

def load_image_as_array(path, target_size=(128, 128)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32)

def classify_images(image_dir: str | Path,classifier: keras.models.Model):
    results = []
    image_dir = Path(image_dir)

    for image_file in tqdm(image_dir.iterdir()):
        image_array = load_image_as_array(str(image_file))
        image_array = np.expand_dims(image_array, axis=0)

        probabilities = classifier.predict(image_array,verbose=0)
        proba_photo = float(probabilities[0][PHOTO_CLASS_INDEX])

        results.append((str(image_file),proba_photo,proba_photo >= THRESHOLD))

    return results

def move_photos_to_folder(classification_results, dest_dir=PHOTOS_DIR):
    os.makedirs(dest_dir, exist_ok=True)
    moved = []
    for path, _, is_photo in classification_results:
        if is_photo:
            if os.path.abspath(os.path.dirname(path)) != os.path.abspath(dest_dir):
                filename = os.path.basename(path)
                dest_path = os.path.join(dest_dir, filename)
                # Un nom COCO identifie une seule image : si elle existe déjà, on l'écrase.
                # Garder le nom intact permet de retrouver l'image_id et évite les doublons.
                os.replace(path, dest_path)
                path = dest_path
            moved.append(path)
    return moved

# Etape 2 : débruitage avec le U-Net du livrable 2
Chaque photo retenue est chargée en RGB, redimensionnée en 299 × 299 et convertie en valeurs entre 0 et 1. Le U-Net du livrable 2 est ensuite appliqué à l’image, et le résultat est sauvegardé dans datasets/denoised_images.

Le U-Net effectue des opérations de pooling qui nécessitent des dimensions compatibles avec des réductions par deux. Le code complète donc temporairement les dimensions de 299 à 300, puis recadre le résultat à 299 × 299 avant de le sauvegarder.

In [6]:
def load_denoiser_model():
    denoiser = keras.models.load_model(DENOISER_MODEL, compile=False)
    return denoiser

def load_image_for_denoiser(path, target_size=(299, 299)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32) / 255.0

def denoise_images(image_paths, denoiser, output_dir=DENOISED_DIR):
    os.makedirs(output_dir, exist_ok=True)

    for path in tqdm(image_paths):
        arr = load_image_for_denoiser(path)
        height, width = arr.shape[:2]
        pad_height = (-height) % 4
        pad_width = (-width) % 4
        padded = np.pad(arr, ((0, pad_height), (0, pad_width), (0, 0)), mode="edge")
        batch = np.expand_dims(padded, axis=0)
        denoised = keras.ops.convert_to_numpy(denoiser(batch, training=False))[0]
        denoised = denoised[:height, :width, :]
        denoised = np.clip(denoised, 0.0, 1.0)

        denoised_img = Image.fromarray(np.rint(denoised * 255).astype("uint8"))
        filename = os.path.basename(path)
        out_path = os.path.join(output_dir, filename)
        denoised_img.save(out_path)
        gc.collect()
        torch.cuda.empty_cache()


# Etape 3: Chargement des données

## Création des vecteurs captions et images

In [7]:
def create_captions_and_path_vectors(annotation_file, image_dir=DENOISED_DIR):
    image_path_to_caption = collections.defaultdict(list)
    train_captions = []
    img_name_vector = []
    mark_start, mark_end = '<start> ', ' <end>'

    with open(annotation_file, 'r') as f:
        annotations = json.load(f)

    for val in annotations['annotations']:
        caption =  mark_start + val['caption'].strip() + mark_end
        image_path = os.path.join(image_dir,'COCO_train2014_' + '%012d.jpg' % (val['image_id']))

        # L'image peut n'exister que sous forme de caractéristiques déjà extraites (.jpg.npy)
        if not (os.path.isfile(image_path) or os.path.isfile(image_path + '.npy')):
            continue
        image_path_to_caption[image_path].append(caption)
        
    train_image_paths = list(image_path_to_caption.keys())


    for image_path in train_image_paths:
        caption_list = image_path_to_caption[image_path]
        train_captions.extend(caption_list)
        [img_name_vector.append(image_path) for _ in range(len(caption_list))]

    return img_name_vector, train_captions

## Prétraitement des annotations

In [8]:
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

def annotation_preprocessing(train_captions, top_k=5000):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,
                                                    oov_token="<unk>",
                                                    filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~')

    tokenizer.fit_on_texts(train_captions)
    tokenizer.word_index['<pad>'] = 0
    tokenizer.index_word[0] = '<pad>'
    train_seqs = tokenizer.texts_to_sequences(train_captions)
    cap_vector = keras.preprocessing.sequence.pad_sequences(train_seqs, padding='post')
    max_length = calc_max_length(train_seqs)

    # Le tokenizer est renvoyé : le décodeur en a besoin pour <start> et pour retrouver les mots
    return cap_vector, max_length, tokenizer

# Etape 4: Extraction des caractéristiques avec InceptionV3

In [9]:
def build_inception_feature_extractor():
    # Charge InceptionV3 pré-entraîné sur ImageNet, sans la tête de classification
    image_model = InceptionV3(weights="imagenet", include_top=False)
    new_input = keras.Input(shape = (299, 299, 3))
    hidden_layer = image_model(new_input)

    return keras.Model(new_input, hidden_layer)

def load_image_for_inception(path, target_size=(299, 299)):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = (img / 127.5) - 1.0
    
    return img, path

def extract_and_save_inception_features(img_name_vector, feature_extractor, batch_size=8):
    encode_train = sorted(set(img_name_vector))

    image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
    image_dataset = image_dataset.map(
    load_image_for_inception, num_parallel_calls=1).batch(batch_size)

    with torch.inference_mode():
        for img, path in tqdm(image_dataset):
            batch_features = feature_extractor(img, training=False)
            batch_features = keras.ops.reshape(batch_features,
                                        (batch_features.shape[0], -1, batch_features.shape[3]))
            for bf, p in zip(batch_features, path):
                bactch_feature = keras.ops.convert_to_numpy(bf)
                path_of_feature = p.numpy().decode("utf-8")
                np.save(path_of_feature, bactch_feature)

            del img
            del path
            del batch_features

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# Etape 5 : séparation train / validation (80/20)

In [10]:
def map_func(img_name, cap):
    img_tensor = np.load(img_name.decode('utf-8')+'.npy')
    return img_tensor, cap

def data_splitting(img_name_vector, cap_vector, buffer_size, batch_size):

    img_to_cap_vector = collections.defaultdict(list)
    for img, cap in zip(img_name_vector, cap_vector):
        img_to_cap_vector[img].append(cap)

    img_keys = list(img_to_cap_vector.keys())
    img_keys_shuffled = shuffle(img_keys)
    slice_index = int(len(img_keys)*0.8)
    img_name_train_keys, img_name_val_keys = img_keys_shuffled[:slice_index], img_keys_shuffled[slice_index:]

    # boucle pour le jeu d'entrainement
    img_name_train = []
    cap_train = []
    for imgt in img_name_train_keys:
        capt_len = len(img_to_cap_vector[imgt])
        img_name_train.extend([imgt] * capt_len)
        cap_train.extend(img_to_cap_vector[imgt])

    # Boucle pour le jeu de test
    img_name_val = []
    cap_val = []
    for imgv in img_name_val_keys:
        capv_len = len(img_to_cap_vector[imgv])
        img_name_val.extend([imgv] * capv_len)
        cap_val.extend(img_to_cap_vector[imgv])

    dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))
    dataset = dataset.map(lambda item1, item2: tf.numpy_function(
            map_func, [item1, item2], [tf.float32, tf.int32]),
            num_parallel_calls=tf.data.experimental.AUTOTUNE)

    # Melanger les donnees et les diviser en batchs
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

    return dataset, img_name_val, cap_val

# Etape 6 : modèle encodeur / attention / décodeur GRU
On reprend l'architecture du workshop captioning :

- **CNN_Encoder** : une couche dense projette chacune des 64 zones de l'image de 2048 à `EMBEDDING_DIM` caractéristiques ;
- **BahdanauAttention** : à partir des zones encodées et de l'état caché du GRU, calcule un poids par zone (softmax) puis le vecteur de contexte, somme pondérée des zones ;
- **RNN_Decoder** : à chaque pas, le mot précédent passe par une couche Embedding, est concaténé au vecteur de contexte, puis le GRU met à jour son état caché. Deux couches denses donnent ensuite un score pour chaque mot du vocabulaire.

Le code utilise `keras.ops` plutôt que `tf.*` pour fonctionner avec le backend Torch choisi en début de notebook.

In [ ]:
BATCH_SIZE = 64          # taille du batch
BUFFER_SIZE = 1000       # taille du buffer pour mélanger les données
EMBEDDING_DIM = 256      # taille des embeddings des mots et des zones de l'image
UNITS = 512              # taille de l'état caché du GRU
TOP_K = 5000             # nombre de mots gardés dans le vocabulaire
VOCAB_SIZE = TOP_K + 1   # + le token <pad>
ATTENTION_FEATURES_SHAPE = 64  # nombre de zones de l'image (grille 8 x 8)

In [ ]:
class CNN_Encoder(keras.Model):
    # Les images sont déjà résumées par InceptionV3 : l'encodeur projette seulement chaque zone
    def __init__(self, embedding_dim):
        super().__init__()
        self.fc = layers.Dense(embedding_dim)

    def call(self, x):
        # (batch, 64, 2048) -> (batch, 64, embedding_dim)
        x = self.fc(x)
        x = keras.ops.relu(x)
        return x


class BahdanauAttention(keras.Model):
    def __init__(self, units):
        super().__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, features, hidden):
        # features : (batch, 64, embedding_dim) | hidden : (batch, units)
        hidden_with_time_axis = keras.ops.expand_dims(hidden, 1)

        # Mélange image + texte : (batch, 64, units)
        attention_hidden_layer = keras.ops.tanh(self.W1(features) + self.W2(hidden_with_time_axis))

        # Un score non normalisé par zone : (batch, 64, 1)
        score = self.V(attention_hidden_layer)

        # Poids d'attention : softmax sur les 64 zones
        attention_weights = keras.ops.softmax(score, axis=1)

        # Vecteur de contexte : somme des zones pondérées -> (batch, embedding_dim)
        context_vector = attention_weights * features
        context_vector = keras.ops.sum(context_vector, axis=1)

        return context_vector, attention_weights


class RNN_Decoder(keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super().__init__()
        self.units = units

        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.gru = layers.GRU(self.units,
                              return_sequences=True,
                              return_state=True,
                              recurrent_initializer="glorot_uniform")
        # Couche dense qui aura pour entrée la sortie du GRU
        self.fc1 = layers.Dense(self.units)
        # Dernière couche dense : un score par mot du vocabulaire
        self.fc2 = layers.Dense(vocab_size)

        self.attention = BahdanauAttention(self.units)

    def call(self, x, features, hidden):
        # x : mot précédent (batch, 1) | features : (batch, 64, embedding_dim) | hidden : (batch, units)
        context_vector, attention_weights = self.attention(features, hidden)

        # Mot courant -> (batch, 1, embedding_dim)
        x = self.embedding(x)

        # Concaténation contexte + mot -> (batch, 1, embedding_dim * 2)
        x = keras.ops.concatenate([keras.ops.expand_dims(context_vector, 1), x], axis=-1)

        # Le GRU repart de l'état caché du pas précédent : c'est sa mémoire des mots déjà écrits
        output, state = self.gru(x, initial_state=hidden)

        # (batch, 1, units) -> (batch, units)
        y = self.fc1(output)
        y = keras.ops.reshape(y, (-1, y.shape[2]))

        # (batch, vocab_size) : scores du mot suivant
        y = self.fc2(y)

        return y, state, attention_weights

    def reset_state(self, batch_size):
        return keras.ops.zeros((batch_size, self.units))

# Pipeline complète
Les caractéristiques étant déjà extraites dans `features_extracted`, la pipeline peut sauter la classification, le débruitage et l'extraction (`use_extracted_features=True`).

In [11]:
photos_path = Path(PHOTOS_DIR)
photos_moved = [os.path.join(photos_path, f.name) for f in photos_path.iterdir() if f.is_file()]

In [ ]:
def run_pipeline(use_extracted_features=True):
    if use_extracted_features:
        # Les caractéristiques InceptionV3 sont déjà dans FEATURES_DIR : on saute les étapes 1, 2 et 4
        image_dir = FEATURES_DIR
    else:
        print(f"1. Classification des images")
        classifier = load_classifier_model()
        classification_results = classify_images(COCO_DIR, classifier)
        photos_moved = move_photos_to_folder(classification_results)
        print(f"{len(photos_moved)} photos deplacees dans {PHOTOS_DIR}.")
        print("2. Débruitage")
        image_dir = DENOISED_DIR

    print("3. Chargement des donnees et pretraitement")
    img_name_vector, train_captions = create_captions_and_path_vectors(ANNOTATION_FILE, image_dir)
    cap_vector, max_length, tokenizer = annotation_preprocessing(train_captions, top_k=TOP_K)

    if not use_extracted_features:
        print("4. Extraction des caracteristiques avec InceptionV3")
        feature_extractor = build_inception_feature_extractor()
        extract_and_save_inception_features(img_name_vector, feature_extractor)
        print(f"Extraction terminee dans : {DENOISED_DIR}")

    print("5. Séparation train/validation")
    dataset, img_name_val, cap_val = data_splitting(img_name_vector, cap_vector,
                                                     buffer_size=BUFFER_SIZE, batch_size=BATCH_SIZE)

    return dataset, img_name_val, cap_val, tokenizer, max_length

In [13]:
dataset, img_name_val, cap_val, tokenizer, max_length = run_pipeline(use_extracted_features=True)

3. Chargement des donnees et pretraitement
4. Extraction des caracteristiques avec InceptionV3


  0%|          | 0/9455 [00:00<?, ?it/s]

Extraction terminee dans : c:\Users\brago\Documents\# Daryl goat\CESI\Leyenda\datasets\Livrable 3\denoised_images


# Etape 7 : entraînement
À chaque batch, l'état caché est remis à zéro et le décodeur reçoit le token `<start>`. Il prédit ensuite la légende mot par mot ; à chaque pas, on lui donne le **vrai** mot de la légende comme entrée suivante (*teacher forcing*). La perte est une entropie croisée qui ignore les tokens `<pad>`.

Le workshop calcule les gradients avec `tf.GradientTape`, qui ne fonctionne pas avec le backend Torch. On utilise donc la méthode Torch : `loss.backward()`, puis `optimizer.apply()` de Keras.

In [ ]:
encoder = CNN_Encoder(EMBEDDING_DIM)
decoder = RNN_Decoder(EMBEDDING_DIM, UNITS, VOCAB_SIZE)

optimizer = keras.optimizers.Adam()
loss_object = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=None)

ENCODER_WEIGHTS = os.path.join(MODELS_DIR, "encoder_captioning.weights.h5")
DECODER_WEIGHTS = os.path.join(MODELS_DIR, "decoder_gru_captioning.weights.h5")

def loss_function(real, pred):
    # Les positions <pad> (indice 0) ne comptent pas dans la perte
    mask = keras.ops.logical_not(keras.ops.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = keras.ops.cast(mask, dtype=loss_.dtype)
    loss_ = loss_ * mask
    return keras.ops.mean(loss_)

def train_step(img_tensor, target):
    loss = 0
    batch_size = target.shape[0]

    # Initialisation de l'état caché pour chaque batch
    hidden = decoder.reset_state(batch_size=batch_size)

    # Le premier mot donné au décodeur est <start>
    dec_input = keras.ops.full((batch_size, 1), tokenizer.word_index["<start>"], dtype="int32")

    features = encoder(img_tensor)

    for i in range(1, target.shape[1]):
        # Prédiction du i-ème mot de chaque légende du batch
        predictions, hidden, _ = decoder(dec_input, features, hidden)
        loss += loss_function(target[:, i], predictions)

        # Teacher forcing : le vrai mot i est l'entrée du pas i + 1
        dec_input = keras.ops.expand_dims(target[:, i], 1)

    total_loss = loss / int(target.shape[1])

    trainable_variables = encoder.trainable_weights + decoder.trainable_weights

    # Calcul des gradients avec Torch, puis mise à jour des poids par l'optimiseur Keras
    encoder.zero_grad()
    decoder.zero_grad()
    loss.backward()
    gradients = [v.value.grad for v in trainable_variables]
    with torch.no_grad():
        optimizer.apply(gradients, trainable_variables)

    return loss, total_loss

In [ ]:
EPOCHS = 20
loss_plot = []

for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0
    num_steps = 0

    for (batch, (img_tensor, target)) in enumerate(dataset):
        # tf.data renvoie des tenseurs TensorFlow : on les convertit pour le backend Torch
        img_tensor = keras.ops.convert_to_tensor(img_tensor.numpy())
        target = keras.ops.convert_to_tensor(target.numpy(), dtype="int32")

        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss += float(keras.ops.convert_to_numpy(t_loss))
        num_steps += 1

        if batch % 100 == 0:
            print("Epoch {} Batch {} Loss {:.4f}".format(
                epoch + 1, batch, float(keras.ops.convert_to_numpy(batch_loss)) / int(target.shape[1])))

    # Sauvegarde de la perte moyenne de l'époque
    loss_plot.append(total_loss / num_steps)

    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        encoder.save_weights(ENCODER_WEIGHTS)
        decoder.save_weights(DECODER_WEIGHTS)

    print("Epoch {} Loss {:.6f}".format(epoch + 1, total_loss / num_steps))
    print("Time taken for 1 epoch {:.1f} sec\n".format(time.time() - start))

# Affichage de la courbe d'entraînement
plt.plot(loss_plot)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Plot")
plt.show()

# Etape 8 : génération de légendes sur le jeu de validation
Le décodeur part de `<start>` et, cette fois, reçoit à chaque pas **le mot qu'il vient de prédire**. Il s'arrête au token `<end>` ou à `max_length` mots. Les poids d'attention de chaque mot sont gardés pour afficher les zones regardées.

Les caractéristiques sont lues directement dans les fichiers `.npy` : InceptionV3 n'est pas rechargé.

In [ ]:
def evaluate(image_path):
    attention_plot = np.zeros((max_length, ATTENTION_FEATURES_SHAPE))

    hidden = decoder.reset_state(batch_size=1)

    # Caractéristiques déjà extraites : (64, 2048) -> (1, 64, 2048)
    img_tensor_val = np.expand_dims(np.load(image_path + ".npy"), 0)

    dec_input = keras.ops.full((1, 1), tokenizer.word_index["<start>"], dtype="int32")
    result = []

    with torch.no_grad():
        features = encoder(img_tensor_val)

        for i in range(max_length):
            predictions, hidden, attention_weights = decoder(dec_input, features, hidden)

            attention_plot[i] = keras.ops.convert_to_numpy(keras.ops.reshape(attention_weights, (-1,)))

            # Tirage du mot suivant selon les probabilités prédites (comme dans le workshop)
            predicted_id = int(keras.ops.convert_to_numpy(keras.random.categorical(predictions, 1))[0][0])
            word = tokenizer.index_word.get(predicted_id, "<unk>")
            result.append(word)

            if word == "<end>":
                return result, attention_plot[:len(result), :]

            dec_input = keras.ops.full((1, 1), predicted_id, dtype="int32")

    return result, attention_plot[:len(result), :]

def plot_attention(image, result, attention_plot):
    temp_image = np.array(Image.open(image))

    fig = plt.figure(figsize=(10, 10))

    len_result = len(result)
    grid_size = int(np.ceil(np.sqrt(len_result)))
    for l in range(len_result):
        temp_att = np.resize(attention_plot[l], (8, 8))
        ax = fig.add_subplot(grid_size, grid_size, l + 1)
        ax.set_title(result[l])
        img = ax.imshow(temp_image)
        ax.imshow(temp_att, cmap="gray", alpha=0.6, extent=img.get_extent())

    plt.tight_layout()
    plt.show()

In [ ]:
# Affichage d'une légende du jeu de validation
rid = np.random.randint(0, len(img_name_val))
image = img_name_val[rid]
real_caption = " ".join([tokenizer.index_word[i] for i in cap_val[rid] if i not in [0]])
result, attention_plot = evaluate(image)

print("Real Caption:", real_caption)
print("Prediction Caption:", " ".join(result))

# L'attention n'est affichée que si l'image d'origine est disponible
original_image = os.path.join(COCO_DIR, os.path.basename(image))
if os.path.isfile(original_image):
    plot_attention(original_image, result, attention_plot)